# 🎓 CCD UNAB — Cuaderno de Consultas SQL
## Centro de Competencias Digitales — Universidad Autónoma de Bucaramanga

---

Este cuaderno documenta las principales consultas SQL utilizadas por el sistema de seguimiento académico del CCD-UNAB.
La base de datos **`cosmos`** gestiona el progreso de los estudiantes en los tres pilares de competencias digitales.

**Servidor:** `unab-n8n.duckdns.org:5432`  
**Base de datos:** `cosmos`  
**Usuario de solo lectura:** `ccd_reader` *(ver sección de configuración)*

---

### 🗂️ Tablas del sistema

| Tabla / Vista | Descripción |
|---|---|
| `estudiante` | Datos maestros del estudiante |
| `catalogo_materias_ccd` | Catálogo de materias por plan y pilar |
| `cursos_estudiantes` | Historial de cursos matriculados |
| `registro_nota` | Registro de calificaciones (`A`=Aprobado, `R`=Reprobado) |
| `oferta` | Oferta vigente de cursos abiertos a inscripción |
| `progreso_estudiante` | **Vista** consolidada de progreso por pilares |

### 🏛️ Los tres pilares del CCD

1. **Competencias Digitales Básicas** — Cultura digital, identidad en línea, ciudadanía digital.
2. **Analítica y Contenido Digital** — Datos, dashboards, marca personal, producción de contenidos.
3. **Transformación Digital** — IA Generativa, Marco Legal Digital, Bienestar Digital.

## ⚙️ 0. Instalación de dependencias y conexión

Instalamos `psycopg2-binary` (adaptador PostgreSQL para Python) y definimos la función de conexión reutilizable.
La contraseña se lee de una variable de entorno para no exponerla en el código.

In [1]:
# Instalar el adaptador de PostgreSQL para Python
!pip install psycopg2-binary pandas --quiet

In [4]:
import psycopg2
import psycopg2.extras
import pandas as pd
from IPython.display import display

# ── Parámetros de conexión ──────────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "cosmos",
    "user":     "ccd_reader",       # Usuario de solo lectura
    "password": "Unab2026*"  # ← Reemplaza con la contraseña real
}

def run_query(sql: str, params=None) -> pd.DataFrame:
    """Ejecuta un SELECT y retorna un DataFrame de Pandas."""
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql, params)
            rows = cur.fetchall()
    return pd.DataFrame(rows)

# Verificar conexión
try:
    df_test = run_query("SELECT current_database(), current_user, now()::date AS hoy;")
    display(df_test)
    print("✅ Conexión exitosa a la base de datos cosmos")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

,current_database,current_user,hoy
0,cosmos,ccd_reader,2026-04-23


✅ Conexión exitosa a la base de datos cosmos


---
## 📋 1. Tabla `estudiante`

### Estructura

| Columna | Tipo | Descripción |
|---|---|---|
| `id_estudiante` | varchar(20) | **PK** — Código único del estudiante (ej: `20250001`) |
| `nombre_completo` | varchar(150) | Nombre completo |
| `programa_academico` | varchar(150) | Carrera que cursa |
| `facultad` | varchar(100) | Facultad a la que pertenece |
| `anio_ingreso` | integer | Año de ingreso a la universidad |
| `semestre_actual` | integer | Semestre en curso |
| `email` | varchar(150) | Correo institucional |
| `telegram_id` | bigint | ID de Telegram para notificaciones |
| `activo` | boolean | Si el estudiante está activo |

### Lógica de planes
- Estudiantes con `anio_ingreso >= 2025` → **Plan Nuevo**  
- Estudiantes con `anio_ingreso < 2025` → **Plan Antiguo**

### Ejemplo de datos
```
id_estudiante | nombre_completo              | programa_academico        | anio_ingreso | semestre_actual
20250001      | Valentina Gómez Rincón       | Ingeniería de Sistemas    | 2025         | 2
20220003      | Laura Cristina Vargas Díaz   | Derecho                   | 2022         | 8
20200008      | Julián Mauricio Pinto García  | Ingeniería Industrial     | 2020         | 10
```

In [ ]:
# ── CONSULTA 1.1: Todos los estudiantes activos ──────────────────────
# Retorna el listado completo de estudiantes registrados en el sistema,
# clasificando automáticamente su plan (nuevo/antiguo) según el año de ingreso.

sql = """
SELECT
    id_estudiante,
    nombre_completo,
    programa_academico,
    facultad,
    anio_ingreso,
    semestre_actual,
    CASE
        WHEN anio_ingreso >= 2025 THEN 'Nuevo'
        ELSE 'Antiguo'
    END AS plan
FROM estudiante
WHERE activo = TRUE
ORDER BY anio_ingreso DESC, nombre_completo;
"""

df = run_query(sql)
print(f"Total de estudiantes activos: {len(df)}")
display(df)

In [ ]:
# ── CONSULTA 1.2: Distribución de estudiantes por plan ──────────────
# Agrupa los estudiantes según su plan (antiguo vs nuevo) para tener
# una visión estadística de cuántos corresponden a cada modalidad curricular.

sql = """
SELECT
    CASE
        WHEN anio_ingreso >= 2025 THEN 'Plan Nuevo (2025+)'
        ELSE 'Plan Antiguo (antes de 2025)'
    END AS plan,
    COUNT(*) AS total_estudiantes,
    MIN(anio_ingreso) AS desde_anio,
    MAX(anio_ingreso) AS hasta_anio
FROM estudiante
WHERE activo = TRUE
GROUP BY plan
ORDER BY plan;
"""

df = run_query(sql)
display(df)

In [ ]:
# ── CONSULTA 1.3: Buscar un estudiante por ID ────────────────────────
# Útil para verificar si un estudiante específico está registrado
# y conocer sus datos básicos. Se puede cambiar el ID de búsqueda.

ID_BUSCAR = "20220003"   # ← Cambia este valor

sql = """
SELECT
    id_estudiante,
    nombre_completo,
    programa_academico,
    facultad,
    anio_ingreso,
    semestre_actual,
    email,
    CASE WHEN anio_ingreso >= 2025 THEN 'Nuevo' ELSE 'Antiguo' END AS plan
FROM estudiante
WHERE id_estudiante = %s;
"""

df = run_query(sql, (ID_BUSCAR,))
if df.empty:
    print(f"⚠️ No se encontró el estudiante con ID {ID_BUSCAR}")
else:
    display(df)

---
## 📚 2. Tabla `catalogo_materias_ccd`

### Estructura

| Columna | Tipo | Descripción |
|---|---|---|
| `codigo_materia` | varchar(20) | **PK** — Código único de la materia |
| `codigo_curso` | varchar(20) | Código del curso asociado en Banner/SIS |
| `nombre_materia` | varchar(150) | Nombre completo de la materia |
| `plan` | varchar(20) | `'antiguo'` o `'nuevo'` |
| `pilar` | varchar(100) | Pilar del CCD al que pertenece |
| `descripcion` | text | Descripción del contenido del curso |

### Ejemplo de datos
```
codigo_materia | nombre_materia              | plan    | pilar
CLDG           | Cultura Digital             | antiguo | Competencias Digitales Básicas
HECO           | Interacción Digital         | nuevo   | Competencias Digitales Básicas
TINF           | IA Generativa               | nuevo   | Transformación Digital
EXCL           | Excel                       | antiguo | Transformación Digital
```

In [ ]:
# ── CONSULTA 2.1: Catálogo completo de materias del CCD ─────────────
# Muestra todas las materias disponibles, agrupadas por pilar y plan,
# lo que permite visualizar la malla curricular completa del CCD.

sql = """
SELECT
    pilar,
    plan,
    codigo_materia,
    codigo_curso,
    nombre_materia,
    descripcion
FROM catalogo_materias_ccd
ORDER BY
    pilar,
    plan DESC,   -- 'nuevo' antes que 'antiguo'
    nombre_materia;
"""

df = run_query(sql)
print(f"Total de materias en catálogo: {len(df)}")
display(df)

In [ ]:
# ── CONSULTA 2.2: Materias agrupadas por pilar ──────────────────────
# Resumen de cuántas materias hay en cada pilar, separadas por plan.
# Útil para validar que la malla curricular esté completa y balanceada.

sql = """
SELECT
    pilar,
    plan,
    COUNT(*) AS cantidad_materias,
    STRING_AGG(nombre_materia, ', ' ORDER BY nombre_materia) AS materias
FROM catalogo_materias_ccd
GROUP BY pilar, plan
ORDER BY pilar, plan;
"""

df = run_query(sql)
display(df)

---
## 📝 3. Tabla `cursos_estudiantes`

### Estructura

| Columna | Tipo | Descripción |
|---|---|---|
| `id` | integer | **PK** — Auto-incremental |
| `id_estudiante` | varchar(20) | FK → `estudiante.id_estudiante` |
| `codigo_materia` | varchar(20) | Código de la materia cursada |
| `codigo_curso` | varchar(20) | Código del grupo/sección |
| `nombre_materia` | varchar(150) | Nombre de la materia |
| `semestre` | varchar(20) | Semestre (ej: `2023-1`) |
| `anio` | integer | Año en que cursó |
| `fecha_matricula` | date | Fecha de matrícula |

### Ejemplo de datos
```
id | id_estudiante | codigo_materia | nombre_materia   | semestre | anio | fecha_matricula
 1 | 20220003      | CLDG           | Cultura Digital  | 2022-2   | 2022 | 2022-08-01
 2 | 20220003      | VLDN           | Vida en Línea    | 2023-1   | 2023 | 2023-01-15
 9 | 20240005      | HECO           | Interacción Dig. | 2024-1   | 2024 | 2024-01-15
```

In [ ]:
# ── CONSULTA 3.1: Historial de cursos de un estudiante ───────────────
# Recupera todos los cursos que un estudiante ha matriculado,
# con información del pilar correspondiente cruzada desde el catálogo.

ID_BUSCAR = "20220003"   # ← Cambia este valor

sql = """
SELECT
    ce.semestre,
    ce.anio,
    ce.codigo_materia,
    ce.nombre_materia,
    cm.pilar,
    cm.plan,
    ce.fecha_matricula
FROM cursos_estudiantes ce
LEFT JOIN catalogo_materias_ccd cm
       ON cm.codigo_materia = ce.codigo_materia
WHERE ce.id_estudiante = %s
ORDER BY ce.anio, ce.semestre;
"""

df = run_query(sql, (ID_BUSCAR,))
print(f"Cursos matriculados por estudiante {ID_BUSCAR}: {len(df)}")
display(df)

In [ ]:
# ── CONSULTA 3.2: Materias más matriculadas ──────────────────────────
# Identifica cuáles son las materias con mayor número de matrículas.
# Útil para detectar demanda y planificar oferta de cupos.

sql = """
SELECT
    ce.codigo_materia,
    ce.nombre_materia,
    cm.pilar,
    cm.plan,
    COUNT(*) AS total_matriculas
FROM cursos_estudiantes ce
LEFT JOIN catalogo_materias_ccd cm ON cm.codigo_materia = ce.codigo_materia
GROUP BY ce.codigo_materia, ce.nombre_materia, cm.pilar, cm.plan
ORDER BY total_matriculas DESC;
"""

df = run_query(sql)
display(df)

---
## 🏆 4. Tabla `registro_nota`

### Estructura

| Columna | Tipo | Descripción |
|---|---|---|
| `id` | integer | **PK** — Auto-incremental |
| `id_estudiante` | varchar(20) | FK → `estudiante.id_estudiante` |
| `codigo_materia` | varchar(20) | Materia evaluada |
| `codigo_curso` | varchar(20) | Grupo/sección del curso |
| `tipo_registro` | varchar(20) | `'curso'` o `'validacion'` |
| `nota` | char(1) | **`A`** = Aprobado · **`R`** = Reprobado |
| `fecha_registro` | date | Fecha de registro de la nota |
| `observacion` | text | Observaciones adicionales |

### Restricciones de integridad
- `tipo_registro` solo acepta `'curso'` o `'validacion'`
- `nota` solo acepta `'A'` (Aprobado) o `'R'` (Reprobado)

### Ejemplo de datos
```
id | id_estudiante | codigo_materia | tipo_registro | nota | fecha_registro
 1 | 20220003      | CLDG           | curso         |  A   | 2022-12-15
 5 | 20220003      | EXCL           | curso         |  A   | 2024-06-15
 9 | 20240005      | HECO           | curso         |  A   | 2024-06-30
```

In [ ]:
# ── CONSULTA 4.1: Notas de un estudiante con detalle del pilar ───────
# Muestra todas las notas de un estudiante, cruzando con el catálogo
# para incluir el nombre del pilar y el plan correspondiente.
# Columna 'nota': A = Aprobado, R = Reprobado.

ID_BUSCAR = "20220003"   # ← Cambia este valor

sql = """
SELECT
    rn.fecha_registro,
    rn.codigo_materia,
    cm.nombre_materia,
    cm.pilar,
    cm.plan,
    rn.tipo_registro,
    rn.nota,
    CASE rn.nota
        WHEN 'A' THEN '✅ Aprobado'
        WHEN 'R' THEN '❌ Reprobado'
    END AS resultado
FROM registro_nota rn
LEFT JOIN catalogo_materias_ccd cm ON cm.codigo_materia = rn.codigo_materia
WHERE rn.id_estudiante = %s
ORDER BY rn.fecha_registro;
"""

df = run_query(sql, (ID_BUSCAR,))
aprobadas = len(df[df['nota'] == 'A'])
reprobadas = len(df[df['nota'] == 'R'])
print(f"Estudiante {ID_BUSCAR} — Aprobadas: {aprobadas} | Reprobadas: {reprobadas}")
display(df)

In [ ]:
# ── CONSULTA 4.2: Materias aprobadas vs reprobadas (resumen global) ──
# Estadística global del sistema: cuántas aprobaciones y reprobaciones
# hay por pilar y plan. Útil para reportes de gestión académica.

sql = """
SELECT
    cm.pilar,
    cm.plan,
    cm.nombre_materia,
    COUNT(*) FILTER (WHERE rn.nota = 'A') AS aprobados,
    COUNT(*) FILTER (WHERE rn.nota = 'R') AS reprobados,
    COUNT(*) AS total
FROM registro_nota rn
LEFT JOIN catalogo_materias_ccd cm ON cm.codigo_materia = rn.codigo_materia
GROUP BY cm.pilar, cm.plan, cm.nombre_materia
ORDER BY cm.pilar, cm.plan, cm.nombre_materia;
"""

df = run_query(sql)
display(df)

---
## 🗓️ 5. Tabla `oferta`

### Estructura

| Columna | Tipo | Descripción |
|---|---|---|
| `id_oferta` | integer | **PK** — Auto-incremental |
| `codigo_curso` | varchar(20) | Código del curso ofertado |
| `nombre_curso` | varchar(150) | Nombre del curso |
| `pilar` | varchar(100) | Pilar del CCD |
| `modalidad` | varchar(50) | Presencial / Virtual / Híbrido |
| `cupos` | integer | Cupos totales |
| `cupos_disponibles` | integer | Cupos aún disponibles |
| `fecha_inicio` | date | Inicio del curso |
| `fecha_fin` | date | Fin del curso |
| `fecha_inicio_matricula` | date | Apertura de inscripciones |
| `fecha_fin_matricula` | date | Cierre de inscripciones |
| `aula` | varchar(50) | Aula o enlace virtual |
| `docente` | varchar(150) | Nombre del docente |
| `activo` | boolean | Si el curso está activo |

> **Nota:** La vista del MCP filtra por `activo = TRUE` y `fecha_fin_matricula >= hoy` para mostrar solo cursos con inscripción vigente.

In [ ]:
# ── CONSULTA 5.1: Oferta vigente (con inscripción abierta hoy) ───────
# Replica exactamente la lógica de la función get_oferta_vigente() del MCP.
# Muestra solo los cursos activos cuyo plazo de matrícula no ha vencido.

sql = """
SELECT
    nombre_curso,
    pilar,
    modalidad,
    cupos,
    cupos_disponibles,
    fecha_inicio,
    fecha_fin,
    fecha_inicio_matricula,
    fecha_fin_matricula,
    aula,
    docente
FROM oferta
WHERE activo = TRUE
  AND (fecha_fin_matricula IS NULL OR fecha_fin_matricula >= CURRENT_DATE)
ORDER BY fecha_inicio_matricula NULLS LAST;
"""

df = run_query(sql)
print(f"Cursos con inscripción vigente hoy: {len(df)}")
display(df)

In [ ]:
# ── CONSULTA 5.2: Disponibilidad de cupos por pilar ─────────────────
# Muestra cuántos cupos totales y disponibles hay por pilar en la oferta
# activa. Ayuda a planificar si hay suficiente capacidad para los estudiantes
# que aún deben completar cada pilar.

sql = """
SELECT
    pilar,
    COUNT(*)                 AS cursos_activos,
    SUM(cupos)               AS cupos_totales,
    SUM(cupos_disponibles)   AS cupos_disponibles,
    SUM(cupos) - SUM(cupos_disponibles) AS cupos_ocupados
FROM oferta
WHERE activo = TRUE
  AND (fecha_fin_matricula IS NULL OR fecha_fin_matricula >= CURRENT_DATE)
GROUP BY pilar
ORDER BY pilar;
"""

df = run_query(sql)
display(df)

---
## 📊 6. Vista `progreso_estudiante`

Esta **vista consolidada** pre-calcula el estado de cada estudiante en los tres pilares del CCD.
Es el equivalente en SQL de la lógica que hace la función `consultar_progreso_estudiante()` del MCP.

### Estructura

| Columna | Tipo | Descripción |
|---|---|---|
| `id_estudiante` | varchar | ID del estudiante |
| `nombre_completo` | varchar | Nombre completo |
| `programa_academico` | varchar | Carrera |
| `anio_ingreso` | integer | Año de ingreso |
| `plan` | text | `'nuevo'` o `'antiguo'` |
| `pilar1_cumplido` | boolean | ¿Completó Competencias Digitales Básicas? |
| `pilar2_cumplido` | boolean | ¿Completó Analítica y Contenido Digital? |
| `pilar3_cumplido` | boolean | ¿Completó Transformación Digital? |
| `cursos_aprobados` | varchar[] | Array de cursos aprobados |
| `cursos_faltantes` | varchar[] | Array de cursos pendientes |

### Ejemplo de datos
```
id_estudiante | nombre                     | plan    | p1  | p2  | p3  | cursos_aprobados
20220003      | Laura Cristina Vargas Díaz | antiguo |  ✅ |  ✅ |  ✅ | {Cultura Digital, Excel, PowerPoint, Vida en Línea, Word}
20230004      | Andrés Felipe Rojas Castro | antiguo |  ✅ |  ❌ |  ❌ | {Cultura Digital, Vida en Línea, Word}
20250001      | Valentina Gómez Rincón     | nuevo   |  ✅ |  ❌ |  ✅ | {IA Generativa, Interacción Digital}
```

In [ ]:
# ── CONSULTA 6.1: Progreso de todos los estudiantes ──────────────────
# Consulta la vista que consolida el estado de cada pilar por estudiante.
# Los TRUE/FALSE indican si el pilar está completado o pendiente.

sql = """
SELECT
    id_estudiante,
    nombre_completo,
    programa_academico,
    anio_ingreso,
    plan,
    pilar1_cumplido  AS p1_digital_basica,
    pilar2_cumplido  AS p2_analitica,
    pilar3_cumplido  AS p3_transformacion,
    (pilar1_cumplido AND pilar2_cumplido AND pilar3_cumplido) AS ruta_completa,
    cursos_aprobados,
    cursos_faltantes
FROM progreso_estudiante
ORDER BY
    (pilar1_cumplido AND pilar2_cumplido AND pilar3_cumplido) DESC,
    nombre_completo;
"""

df = run_query(sql)
completos = len(df[df['ruta_completa'] == True])
print(f"Total estudiantes: {len(df)} | Ruta completa: {completos} | Pendientes: {len(df)-completos}")
display(df)

In [ ]:
# ── CONSULTA 6.2: Progreso detallado de un estudiante específico ─────
# Replica la lógica central del MCP: consulta el progreso individual
# de un estudiante con su ID, mostrando cada pilar y qué le falta.
# Este es el query que alimenta al chatbot de Telegram.

ID_BUSCAR = "20230004"   # ← Cambia este valor

sql = """
SELECT
    e.id_estudiante,
    e.nombre_completo,
    e.programa_academico,
    e.anio_ingreso,
    CASE WHEN e.anio_ingreso >= 2025 THEN 'Nuevo' ELSE 'Antiguo' END AS plan,
    -- Pilar 1
    pe.pilar1_cumplido,
    -- Pilar 2
    pe.pilar2_cumplido,
    -- Pilar 3
    pe.pilar3_cumplido,
    -- Detalle
    pe.cursos_aprobados,
    pe.cursos_faltantes
FROM progreso_estudiante pe
JOIN estudiante e ON e.id_estudiante = pe.id_estudiante
WHERE pe.id_estudiante = %s;
"""

df = run_query(sql, (ID_BUSCAR,))
if df.empty:
    print(f"⚠️ No se encontró el estudiante con ID {ID_BUSCAR}")
else:
    row = df.iloc[0]
    print(f"\n👤 {row['nombre_completo']} ({row['id_estudiante']})")
    print(f"📚 {row['programa_academico']} — Plan {row['plan']}")
    print(f"\n🔹 Pilar 1 - Competencias Digitales Básicas: {'✅' if row['pilar1_cumplido'] else '❌'}")
    print(f"🔹 Pilar 2 - Analítica y Contenido Digital:  {'✅' if row['pilar2_cumplido'] else '❌'}")
    print(f"🔹 Pilar 3 - Transformación Digital:         {'✅' if row['pilar3_cumplido'] else '❌'}")
    print(f"\n✅ Aprobados: {row['cursos_aprobados']}")
    print(f"❌ Faltantes: {row['cursos_faltantes']}")
    display(df)

In [ ]:
# ── CONSULTA 6.3: Estadísticas de cumplimiento por pilar ─────────────
# Resumen ejecutivo: cuántos estudiantes han completado cada pilar.
# Muestra el porcentaje de cumplimiento para reportes de gestión.

sql = """
SELECT
    plan,
    COUNT(*)                                          AS total,
    COUNT(*) FILTER (WHERE pilar1_cumplido)           AS p1_cumplido,
    COUNT(*) FILTER (WHERE pilar2_cumplido)           AS p2_cumplido,
    COUNT(*) FILTER (WHERE pilar3_cumplido)           AS p3_cumplido,
    COUNT(*) FILTER (WHERE pilar1_cumplido
                       AND pilar2_cumplido
                       AND pilar3_cumplido)           AS ruta_completa,
    ROUND(100.0 * COUNT(*) FILTER (WHERE pilar1_cumplido
                                     AND pilar2_cumplido
                                     AND pilar3_cumplido)
          / NULLIF(COUNT(*), 0), 1)                   AS pct_completado
FROM progreso_estudiante
GROUP BY plan
ORDER BY plan;
"""

df = run_query(sql)
display(df)

---
## 🤖 7. Query principal del MCP

Este es el query central que utiliza la función `consultar_progreso_estudiante()` del servidor MCP.
Combina `registro_nota` con `catalogo_materias_ccd` para que el agente pueda determinar
qué pilares ha completado el estudiante y qué le falta cursar.

In [ ]:
# ── CONSULTA 7.1: Query central del MCP (notas + catálogo) ───────────
# Replica el JOIN exacto que usa la función consultar_progreso_estudiante().
# Cada fila = una materia cursada con su pilar y si fue aprobada (A) o no.
# Con este resultado el agente evalúa los pilares y construye la respuesta.

ID_BUSCAR = "20220003"   # ← Cambia este valor

sql = """
SELECT
    rn.codigo_materia,
    rn.nota,
    rn.tipo_registro,
    rn.fecha_registro,
    cm.plan,
    cm.pilar,
    cm.nombre_materia
FROM registro_nota rn
LEFT JOIN catalogo_materias_ccd cm
       ON cm.codigo_materia = rn.codigo_materia
WHERE rn.id_estudiante = %s
ORDER BY cm.pilar, rn.nota;
"""

df = run_query(sql, (ID_BUSCAR,))
print(f"Registros encontrados para {ID_BUSCAR}: {len(df)}")
display(df)

In [ ]:
# ── CONSULTA 7.2: Estudiantes con pilares pendientes y oferta vigente ─
# Identifica estudiantes con pilares incompletos Y qué cursos tienen
# disponibles para completarlos. Es el cruce que justificaría
# enviarles una notificación proactiva de inscripción.

sql = """
SELECT
    pe.id_estudiante,
    pe.nombre_completo,
    pe.plan,
    -- Pilares pendientes
    CASE WHEN NOT pe.pilar1_cumplido THEN 'Competencias Digitales Básicas' END   AS pilar1_pendiente,
    CASE WHEN NOT pe.pilar2_cumplido THEN 'Analítica y Contenido Digital'  END   AS pilar2_pendiente,
    CASE WHEN NOT pe.pilar3_cumplido THEN 'Transformación Digital'          END   AS pilar3_pendiente,
    -- Cursos vigentes disponibles (agrupados)
    ARRAY(
        SELECT o.nombre_curso
        FROM oferta o
        WHERE o.activo = TRUE
          AND (o.fecha_fin_matricula IS NULL OR o.fecha_fin_matricula >= CURRENT_DATE)
          AND o.cupos_disponibles > 0
          AND (
              (NOT pe.pilar1_cumplido AND o.pilar = 'Competencias Digitales Básicas') OR
              (NOT pe.pilar2_cumplido AND o.pilar = 'Analítica y Contenido Digital')  OR
              (NOT pe.pilar3_cumplido AND o.pilar = 'Transformación Digital')
          )
        ORDER BY o.nombre_curso
    ) AS cursos_disponibles_para_inscribir
FROM progreso_estudiante pe
WHERE NOT (pe.pilar1_cumplido AND pe.pilar2_cumplido AND pe.pilar3_cumplido)
ORDER BY pe.nombre_completo;
"""

df = run_query(sql)
print(f"Estudiantes con ruta incompleta: {len(df)}")
display(df)

---
## 📌 Resumen de tablas y relaciones

```
estudiante (id_estudiante PK)
    │
    ├──▶ cursos_estudiantes (id_estudiante FK)
    │         └── codigo_materia ──▶ catalogo_materias_ccd (codigo_materia PK)
    │
    └──▶ registro_nota (id_estudiante FK)
              └── codigo_materia ──▶ catalogo_materias_ccd

oferta (id_oferta PK) — tabla independiente con cupos disponibles

progreso_estudiante — VISTA que consolida registro_nota + catalogo_materias_ccd
```

---
*Cuaderno generado para el CCD-UNAB · Base de datos `cosmos` en `unab-n8n.duckdns.org`*